# Smart Scientific Python

This notebook is about **writing Python in a way that uses the strengths of the ecosystem**.

The main principle is:

> **Do not optimize Python by writing more complicated Python loops.  
> First ask whether NumPy, pandas, SciPy, or another optimized library can do the heavy work for you.**

We will cover:

1. Measuring performance correctly
2. Pure Python vs NumPy
3. Vectorization
4. Broadcasting
5. Boolean masking
6. Reductions and the meaning of `axis`
7. Data types and memory
8. Views, copies, and in-place operations
9. Chunking large computations
10. Linear algebra
11. Random-number generation
12. pandas for tabular data
13. SciPy for optimization and scientific algorithms
14. Sparse matrices
15. Threads, the GIL, and native libraries
16. Multiprocessing
17. A practical optimization workflow

# 1. Setup

In [1]:
import os
import time
import timeit

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("CPUs:", os.cpu_count())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)

CPUs: 8
NumPy: 2.5.2
pandas: 3.0.6


# 2. Measure before optimizing

Before changing code, measure it.

For simple experiments, `time.perf_counter()` is convenient:

In [2]:
start = time.perf_counter()

total = 0
for i in range(1_000_000):
    total += i

elapsed = time.perf_counter() - start

print(f"Elapsed: {elapsed:.4f} s")

Elapsed: 0.0553 s


For more reliable micro-benchmarks, use `timeit`, which repeats the computation several times.

In [3]:
import timeit

def my_function():
    total = 0
    for i in range(1_000_000):
        total += i
    return total
        

t = timeit.timeit(
    my_function,
    number=10
)

print(f"10 runs: {t:.4f} s")
print(f"Average : {t/10:.4f} s")

10 runs: 0.2165 s
Average : 0.0216 s


In [4]:
t = timeit.timeit(
    "sum(range(1_000_000))",
    number=10
)

print(f"10 runs: {t:.4f} s")
print(f"Average : {t/10:.4f} s")

10 runs: 0.0721 s
Average : 0.0072 s


# 3. Pure Python vs NumPy

Consider the same CPU-bound numerical task:

```python
(i % 7) * (i % 13)
```

for many values.

In [5]:
N = 5_000_000

In [6]:
def pure_python(n):
    total = 0
    for i in range(n):
        total += (i % 7) * (i % 13)
    return total

start = time.perf_counter()
result_python = pure_python(N)
time_python = time.perf_counter() - start

print("Result:", result_python)
print(f"Pure Python: {time_python:.4f} s")

Result: 89999940
Pure Python: 0.2086 s


In [7]:
start = time.perf_counter()

x = np.arange(N, dtype=np.int64)
result_numpy = np.sum((x % 7) * (x % 13))

time_numpy = time.perf_counter() - start

print("Result:", int(result_numpy))
print(f"NumPy: {time_numpy:.4f} s")
print(f"Speedup: {time_python / time_numpy:.2f}x")

Result: 89999940
NumPy: 0.0295 s
Speedup: 7.08x


## Why is NumPy faster?

Pure Python:

```text
for each element
    ↓
Python interpreter
    ↓
Python integer operations
    ↓
repeat millions of times
```

NumPy:

```text
one high-level Python operation
    ↓
optimized compiled implementation
    ↓
operate on many values efficiently
```

The key point is that NumPy moves most of the work **out of the Python interpreter**.

# 4. Vectorization

A common optimization pattern is replacing explicit Python loops with array expressions.

In [8]:
x_list = list(range(100_000_000))

start = time.perf_counter()

y_loop = []
for x in x_list:
    y_loop.append(3*x**2 + 2*x + 1)

loop_time = time.perf_counter() - start

print(f"Python loop: {loop_time:.4f} s")

Python loop: 13.5087 s


In [9]:
x = np.arange(100_000_000)

start = time.perf_counter()

y_numpy = 3*x**2 + 2*x + 1

numpy_time = time.perf_counter() - start

print(f"NumPy: {numpy_time:.4f} s")
print(f"Speedup: {loop_time / numpy_time:.2f}x")

NumPy: 1.0939 s
Speedup: 12.35x


Vectorization is especially useful when the **same numerical operation** is applied **independently to many elements**.

# 5. Broadcasting

Broadcasting lets NumPy combine arrays with compatible shapes **without explicitly copying data**.

In [10]:
X = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
])

offset = np.array([10, 20, 30])

print(X + offset)

[[11 22 33]
 [14 25 36]
 [17 28 39]]


Conceptually:

```text
X                 offset

1 2 3             10 20 30
4 5 6      +      10 20 30
7 8 9             10 20 30
```

NumPy behaves *as if* `offset` had been repeated for every row, but it does not need to physically create those copies.

In [11]:
## Here the vector has shape (3, 1) rather than (3,)
X = np.array([
[1, 2, 3],
[4, 5, 6],
[7, 8, 9]
])

row_offset = np.array([
[10],
[20],
[30]
])

print(X + row_offset)

[[11 12 13]
 [24 25 26]
 [37 38 39]]


Conceptually,

```text
1 2 3      10 10 10
4 5 6   +  20 20 20
7 8 9      30 30 30


In [12]:
# Multiply each column by a different factor
# Very common in scientific computing and ML:
X = np.array([
    [1, 2, 3],
    [4, 5, 6]
])

weights = np.array([1, 10, 100])

print(X * weights)

[[  1  20 300]
 [  4  50 600]]


## Row vector + column vector

Broadcasting happens in two directions:

In [13]:
row = np.array([10, 20, 30])
column = np.array([
    [1],
    [2],
    [3],
    [4]
])

result = column + row

print(result)
print(result.shape)

[[11 21 31]
 [12 22 32]
 [13 23 33]
 [14 24 34]]
(4, 3)


## Pairwise differences

Broadcasting is also useful for computing all pairwise combinations.

In [14]:
x = np.array([2, 5, 9])

differences = x[:, None] - x[None, :]

print(differences)

[[ 0 -3 -7]
 [ 3  0 -4]
 [ 7  4  0]]


In [15]:
print(x[:,None])
print(x[None,:])

[[2]
 [5]
 [9]]
[[2 5 9]]


In [16]:
## Pairwise Euclidean distances in 2D

points = np.array([
    [0, 0],
    [1, 0],
    [1, 1]
])

diff = points[:, None, :] - points[None, :, :]

distances = np.sqrt(
    np.sum(diff**2, axis=2)
)

print(distances)

[[0.         1.         1.41421356]
 [1.         0.         1.        ]
 [1.41421356 1.         0.        ]]


## ML-style example: standardization

In [17]:
rng = np.random.default_rng(42)

X = rng.normal(size=(1000, 4))

mean = X.mean(axis=0)
std = X.std(axis=0)

X_std = (X - mean) / std

print("Means:", X_std.mean(axis=0))
print("Std dev:", X_std.std(axis=0))

Means: [ 9.99200722e-18 -2.21489493e-17  5.21804822e-18  2.04281037e-17]
Std dev: [1. 1. 1. 1.]


# 6. Boolean masking

Instead of manually filtering values with a Python loop, NumPy can evaluate conditions on entire arrays.

In [18]:
rng = np.random.default_rng(0)
x = rng.random(100_000_000)

start = time.perf_counter()

selected_loop = []
for value in x:
    if value > 0.8:
        selected_loop.append(value)

loop_filter_time = time.perf_counter() - start

print(f"Loop filtering: {loop_filter_time:.4f} s")

Loop filtering: 6.4682 s


In [19]:
start = time.perf_counter()

selected_numpy = x[x > 0.8]

mask_time = time.perf_counter() - start

print(f"Masking: {mask_time:.4f} s")
print(f"Speedup: {loop_filter_time / mask_time:.2f}x")

Masking: 0.4186 s
Speedup: 15.45x


Conditions can be combined:

In [20]:
selected = x[(x > 0.2) & (x < 0.4)]

print(selected[:10])
print("Count:", selected.size)

[0.26978671 0.29971189 0.38367755 0.38892142 0.31024188 0.3577952
 0.32186939 0.33791123 0.391619   0.22715759]
Count: 20001443


# 7. Reductions and `axis`

NumPy provides efficient reductions such as:

- `sum`
- `mean`
- `min`
- `max`
- `std`
- `argmin`
- `argmax`

In [21]:
X = np.array([
    [1, 2, 3],
    [4, 5, 6]
])

print("Matrix:")
print(X)

print("\nMean over axis=0:", X.mean(axis=0))
print("Mean over axis=1:", X.mean(axis=1))

Matrix:
[[1 2 3]
 [4 5 6]]

Mean over axis=0: [2.5 3.5 4.5]
Mean over axis=1: [2. 5.]


A useful mental model:

```text
axis=0 → collapse rows → one result per column
axis=1 → collapse columns → one result per row
```

# 8. Data types and memory

NumPy arrays have a fixed `dtype`.

This affects both speed and memory.

In [22]:
a64 = np.arange(1_000_000, dtype=np.int64)
a32 = np.arange(1_000_000, dtype=np.int32)

print("int64:", a64.nbytes / 1e6, "MB")
print("int32:", a32.nbytes / 1e6, "MB")

int64: 8.0 MB
int32: 4.0 MB


A smaller `dtype` can reduce memory pressure, but only if it can represent the required values safely.

# 8A. Python lists vs NumPy arrays

NumPy arrays are powerful, but they are **not a replacement for Python lists in every situation**.

A Python list is a general-purpose container.

A NumPy array is designed primarily for **homogeneous numerical data and efficient array computation**.

## Lists can contain heterogeneous objects

A Python list can contain objects of completely different types:

In [23]:
mixed_list = [
    42,
    "hello",
    3.14,
    True,
    {"name": "Anna"}
]

print(mixed_list)

[42, 'hello', 3.14, True, {'name': 'Anna'}]


A NumPy array normally tries to use **one common `dtype`** for all its elements.

For example:

In [24]:
a = np.array([1, 2, 3.5])

print(a)
print("dtype:", a.dtype)

[1.  2.  3.5]
dtype: float64


The integers are converted to floating-point values because the array needs one compatible numerical type.

NumPy *can* store arbitrary Python objects using `dtype=object`, but then many of the normal performance advantages of NumPy disappear.

In [25]:
a = np.array([1, "hello", {"x": 3}], dtype=object)

print(a)
print(a.dtype)

[1 'hello' {'x': 3}]
object


## Lists are dynamically resizable

Python lists are designed to grow and shrink:

In [26]:
values = [1, 2, 3]
print(values)

values.append(4)
print(values)

values.insert(0, 0)
print(values)

values.pop()
print(values)

[1, 2, 3]
[1, 2, 3, 4]
[0, 1, 2, 3, 4]
[0, 1, 2, 3]


NumPy arrays have a **fixed size** after creation.

Functions such as:

```python
np.append(...)
np.insert(...)
np.delete(...)
```

do not usually modify the existing array in place.

They create a **new array**.

In [27]:
a = np.array([1, 2, 3])

b = np.append(a, 4)

print("a:", a)
print("b:", b)
print("Same object?", a is b)

a: [1 2 3]
b: [1 2 3 4]
Same object? False


This means repeatedly doing:

```python
for value in data:
    array = np.append(array, value)
```

is usually inefficient.

If the number of elements is not known in advance, a better strategy is often:

```text
build a Python list
        ↓
append values efficiently
        ↓
convert once to np.array
```

In [28]:
values = []

for i in range(10):
    values.append(i**2)

a = np.array(values)

print(a)

[ 0  1  4  9 16 25 36 49 64 81]


## Lists support general Python operations naturally

Lists are often better when:

- the elements are heterogeneous objects;
- the collection changes size frequently;
- you mostly append or remove elements;
- numerical vectorized operations are not important;
- you want a general-purpose Python container.

NumPy arrays are usually better when:

- the data is homogeneous and numerical;
- the shape is regular;
- you perform mathematical operations on many elements;
- memory efficiency matters;
- vectorization, broadcasting, or linear algebra is needed.

## Important difference: `+`

With Python lists, `+` means **concatenation**:

In [29]:
a = [1, 2, 3]
b = [10, 20, 30]

print(a + b)

[1, 2, 3, 10, 20, 30]


With NumPy arrays, `+` means **element-wise addition**:

In [30]:
a = np.array([1, 2, 3])
b = np.array([10, 20, 30])

print(a + b)

[11 22 33]


To concatenate NumPy arrays explicitly:

In [31]:
print(np.concatenate([a, b]))

[ 1  2  3 10 20 30]


In [32]:
print('hstack: ', np.hstack([a, b]))
print('vstack: ',np.vstack([a, b]))
print('stack: ',np.stack([a, b]))

hstack:  [ 1  2  3 10 20 30]
vstack:  [[ 1  2  3]
 [10 20 30]]
stack:  [[ 1  2  3]
 [10 20 30]]


## Another important difference: multiplication

For a list:

In [33]:
print([1, 2, 3] * 3)

[1, 2, 3, 1, 2, 3, 1, 2, 3]


This repeats the list.

For a NumPy array:

In [34]:
print(np.array([1, 2, 3]) * 3)

[3 6 9]


This multiplies every numerical element by 3.

So the same operators can have very different meanings.

## Summary: list or NumPy array?

| Feature | Python list | NumPy array |
|---|---|---|
| Mixed element types | Yes | Usually no |
| Easy append/remove | Yes | No, usually requires new array |
| Vectorized arithmetic | No | Yes |
| Broadcasting | No | Yes |
| Linear algebra | No | Yes |
| Memory-efficient numeric storage | Less | More |
| General Python objects | Excellent | Possible, but not ideal |
| Fixed multidimensional shape | Not required | Natural |

# 9. Views vs copies

Many NumPy slices return **views**, not independent copies.

In [35]:
x = np.arange(10)

y = x[2:6]

print("x:", x)
print("y:", y)

y[0] = 999

print("\nAfter modifying y:")
print("x:", x)
print("y:", y)

x: [0 1 2 3 4 5 6 7 8 9]
y: [2 3 4 5]

After modifying y:
x: [  0   1 999   3   4   5   6   7   8   9]
y: [999   3   4   5]


If you need independent data, call `.copy()` explicitly.

In [36]:
x = np.arange(10)

y = x[2:6].copy()
y[0] = 999

print("x:", x)
print("y:", y)

x: [0 1 2 3 4 5 6 7 8 9]
y: [999   3   4   5]


# 10. In-place operations

Expressions can create temporary arrays.

For large data, temporary allocations may matter.

In [37]:
x = np.arange(5_000_000, dtype=np.float64)

x2 = x.copy()
print(x2[:5])

x2 *= 2
print(x2[:5])

x2 += 3
print(x2[:5])

[0. 1. 2. 3. 4.]
[0. 2. 4. 6. 8.]
[ 3.  5.  7.  9. 11.]


In-place operations such as:

```python
x *= 2
x += 3
```

can reduce memory allocations.

Use them carefully because they modify the original array.

# 11. Chunking large computations

Vectorization is fast, but a full array may be too large for memory.

Chunking processes smaller blocks one at a time.

In [38]:
def chunked_sum(n, chunk_size=1_000_000):
    total = 0

    for start_idx in range(0, n, chunk_size):
        stop_idx = min(start_idx + chunk_size, n)

        x = np.arange(start_idx, stop_idx, dtype=np.int64)

        total += np.sum((x % 7) * (x % 13))

    return total

In [39]:
start = time.perf_counter()

result_chunked = chunked_sum(
    20_000_000,
    chunk_size=1_000_000
)

chunk_time = time.perf_counter() - start

print("Result:", int(result_chunked))
print(f"Chunked NumPy: {chunk_time:.4f} s")

Result: 359999961
Chunked NumPy: 0.1320 s


Chunking is useful because it:

- reduces peak memory usage;
- lets you process datasets larger than RAM;
- creates natural independent units of work;
- makes later parallelization easier.

Independent chunks can be assigned to different worker processes.

# 12. Linear algebra

Use optimized linear algebra instead of Python loops.

In [40]:
rng = np.random.default_rng(42)

A = rng.normal(size=(500, 500))
B = rng.normal(size=(500, 500))

start = time.perf_counter()

C = A @ B

elapsed = time.perf_counter() - start

print(C.shape)
print(f"Matrix multiplication: {elapsed:.4f} s")

(500, 500)
Matrix multiplication: 0.0065 s


The `@` operator delegates matrix multiplication to optimized numerical libraries.

Depending on your NumPy installation, those libraries may internally use multiple CPU cores.

# 12A. A practical NumPy linear algebra toolkit

NumPy provides a substantial set of linear-algebra operations through:

```python
np.linalg
```

For numerical work, these should generally be preferred over implementing algorithms manually.

## Vector dot product

For two vectors:

In [41]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

print("np.dot:", np.dot(a, b))
print("@     :", a @ b)

np.dot: 32
@     : 32


For one-dimensional arrays, both compute:

\[
a^	op b
\]

The `@` operator is often the clearest notation for matrix/vector multiplication.

## Matrix-vector multiplication

In [42]:
A = np.array([
    [1, 2],
    [3, 4]
])

x = np.array([5, 6])

print(A @ x)

[17 39]


## Matrix-matrix multiplication

In [43]:
A = np.array([
    [1, 2],
    [3, 4]
])

B = np.array([
    [5, 6],
    [7, 8]
])

print(A @ B)

[[19 22]
 [43 50]]


Be careful:

```python
A * B
```

is **element-wise multiplication**, not matrix multiplication.

In [44]:
print("Element-wise:")
print(A * B)

print("\nMatrix multiplication:")
print(A @ B)

Element-wise:
[[ 5 12]
 [21 32]]

Matrix multiplication:
[[19 22]
 [43 50]]


### `@` vs `np.dot()` in higher dimensions

For 2D arrays, `@` and `np.dot()` behave similarly.

For higher-dimensional arrays:

- `@` performs **batched matrix multiplication**
- `np.dot()` contracts the last axis of the first array with the second-last axis of the second array

In [45]:
A = np.ones((32, 3, 4))
B = np.ones((32, 4, 5))

print((A @ B).shape)
print(np.dot(A, B).shape)

(32, 3, 5)
(32, 3, 32, 5)


`@` pairs corresponding batches:

`A[0] @ B[0]`, `A[1] @ B[1]`

So:

`(32, 3, 4) @ (32, 4, 5) -> (32, 3, 5)`

`@` also supports broadcasting over batch dimensions.

**Rule of thumb:** use `@` for matrix multiplication, especially with higher-dimensional arrays.

`np.dot()` does **not** pair corresponding batches.

It combines each batch of `A` with each batch of `B`:

`A[0]` with `B[0]`, `A[0]` with `B[1]`,  
`A[1]` with `B[0]`, `A[1]` with `B[1]`

So:

`np.dot((2, 3, 4), (2, 4, 5)) -> (2, 3, 2, 5)`

## Transpose

In [46]:
A = np.array([
    [1, 2, 3],
    [4, 5, 6]
])

print(A)
print("\nTranspose:")
print(A.T)

[[1 2 3]
 [4 5 6]]

Transpose:
[[1 4]
 [2 5]
 [3 6]]


## Vector and matrix norms

Norms measure magnitude.

A common vector norm is the Euclidean norm:

In [47]:
x = np.array([3.0, 4.0])

print(np.linalg.norm(x))

5.0


You can also specify different norms:

In [48]:
x = np.array([1.0, -2.0, 3.0])

print("L1 norm:", np.linalg.norm(x, ord=1))
print("L2 norm:", np.linalg.norm(x, ord=2))
print("L∞ norm:", np.linalg.norm(x, ord=np.inf))

L1 norm: 6.0
L2 norm: 3.7416573867739413
L∞ norm: 3.0


## Solving linear systems

Suppose:

\[
Ax=b
\]

Do **not** normally compute the inverse explicitly.

Prefer:

In [49]:
A = np.array([
    [3.0, 1.0],
    [1.0, 2.0]
])

b = np.array([9.0, 8.0])

x = np.linalg.solve(A, b)

print("Solution:", x)
print("Check A @ x:", A @ x)

Solution: [2. 3.]
Check A @ x: [9. 8.]


A common beginner approach is:

```python
x = np.linalg.inv(A) @ b
```

but `np.linalg.solve(A, b)` is generally preferable because it is:

- clearer;
- more numerically stable;
- usually more efficient.

## Matrix inverse

The inverse is still available when you genuinely need it:

In [50]:
A_inv = np.linalg.inv(A)

print(A_inv)
print("\nA @ A_inv:")
print(A @ A_inv)

[[ 0.4 -0.2]
 [-0.2  0.6]]

A @ A_inv:
[[1. 0.]
 [0. 1.]]


## Determinant

In [51]:
print("det(A) =", np.linalg.det(A))

det(A) = 5.000000000000001


A determinant close to zero may indicate that a matrix is close to singular, but for numerical analysis it is often better to look at quantities such as the **condition number**.

## Condition number

The condition number tells us how sensitive a problem can be to small perturbations.

In [52]:
print("Condition number:", np.linalg.cond(A))

Condition number: 2.618033988749896


A very large condition number indicates an **ill-conditioned** matrix.

In that case, small errors in the data or floating-point arithmetic can produce large changes in the solution.

## Eigenvalues and eigenvectors

In [53]:
A = np.array([
    [2.0, 1.0],
    [1.0, 2.0]
])

eigenvalues, eigenvectors = np.linalg.eig(A)

print("Eigenvalues:")
print(eigenvalues)

print("\nEigenvectors:")
print(eigenvectors)

Eigenvalues:
[3.+0.j 1.+0.j]

Eigenvectors:
[[ 0.70710678+0.j -0.70710678+0.j]
 [ 0.70710678+0.j  0.70710678+0.j]]


For a real symmetric matrix, prefer:

```python
np.linalg.eigh(...)
```

because it exploits the matrix structure and returns real-valued eigenpairs more reliably.

In [54]:
eigenvalues, eigenvectors = np.linalg.eigh(A)

print(eigenvalues)
print(eigenvectors)

[1. 3.]
[[-0.70710678  0.70710678]
 [ 0.70710678  0.70710678]]


## Singular Value Decomposition (SVD)

SVD factors a matrix as:

\[
A = U \Sigma V^	op
\]

It is fundamental in:

- PCA;
- dimensionality reduction;
- low-rank approximation;
- numerical linear algebra;
- pseudoinverses.

In [55]:
A = np.array([
    [3.0, 1.0],
    [1.0, 3.0],
    [1.0, 1.0]
])

U, singular_values, Vt = np.linalg.svd(A, full_matrices=False)

print("U:")
print(U)

print("\nSingular values:")
print(singular_values)

print("\nV^T:")
print(Vt)

U:
[[-6.66666667e-01  7.07106781e-01]
 [-6.66666667e-01 -7.07106781e-01]
 [-3.33333333e-01  5.92577380e-18]]

Singular values:
[4.24264069 2.        ]

V^T:
[[-0.70710678 -0.70710678]
 [ 0.70710678 -0.70710678]]


## Least-squares problems

If a linear system has more equations than unknowns, there may be no exact solution.

NumPy can compute the least-squares solution:

In [56]:
A = np.array([
    [1.0, 1.0],
    [1.0, 2.0],
    [1.0, 3.0],
    [1.0, 4.0]
])

b = np.array([2.2, 2.8, 3.6, 4.5])

solution, residuals, rank, singular_values = np.linalg.lstsq(
    A,
    b,
    rcond=None
)

print("Solution:", solution)
print("Residuals:", residuals)

Solution: [1.35 0.77]
Residuals: [0.023]


This is the numerical core of ordinary linear regression.

Here the two unknown parameters can be interpreted as an intercept and a slope.

## Pseudoinverse

For rectangular or singular matrices:

In [57]:
A_pinv = np.linalg.pinv(A)

print(A_pinv.shape)

(2, 4)


The Moore-Penrose pseudoinverse is useful conceptually, although in many numerical problems specialized solvers such as `lstsq` are preferable to explicitly forming the pseudoinverse.

## Useful linear algebra functions at a glance

```text
A @ B                 matrix multiplication
np.dot(a, b)          dot product / related products
A.T                    transpose
np.linalg.norm(x)      norms
np.linalg.solve(A,b)   solve Ax=b
np.linalg.inv(A)       matrix inverse
np.linalg.det(A)       determinant
np.linalg.cond(A)      condition number
np.linalg.eig(A)       eigenvalues/eigenvectors
np.linalg.eigh(A)      symmetric/Hermitian eigenproblem
np.linalg.svd(A)       singular value decomposition
np.linalg.lstsq(A,b)   least-squares solution
np.linalg.pinv(A)      pseudoinverse
```

For more specialized matrix algorithms, decompositions, and sparse linear algebra, **SciPy** provides an even larger toolbox in:

```python
scipy.linalg
scipy.sparse.linalg
```

# 13. Random-number generation

Prefer NumPy's modern random generator:

In [58]:
rng = np.random.default_rng(123)

samples = rng.normal(
    loc=0,
    scale=1,
    size=100_000
)

print(samples[:5])

[-0.98912135 -0.36778665  1.28792526  0.19397442  0.9202309 ]


This gives:

- explicit random state;
- reproducibility;
- cleaner scientific workflows.

# 14. pandas: use column operations, not row loops

pandas is designed for labeled tabular data.

In [59]:
df = pd.DataFrame({
    "age": [25, 41, 67, 52],
    "weight": [60, 72, 80, 65],
    "height": [1.65, 1.78, 1.72, 1.60]
})

df

,age,weight,height
0,25,60,1.65
1,41,72,1.78
2,67,80,1.72
3,52,65,1.60


In [60]:
df["bmi"] = df["weight"] / (df["height"] ** 2)

df

,age,weight,height,bmi
0,25,60,1.65,22.038567
1,41,72,1.78,22.724403
2,67,80,1.72,27.041644
3,52,65,1.60,25.390625


Filtering:

In [61]:
df[df["age"] >= 50]

,age,weight,height,bmi
2,67,80,1.72,27.041644
3,52,65,1.60,25.390625


Grouping:

In [62]:
df2 = pd.DataFrame({
    "group": ["A", "A", "B", "B", "B"],
    "value": [10, 12, 15, 14, 18]
})

df2.groupby("group")["value"].mean()

group
A    11.000000
B    15.666667
Name: value, dtype: float64

Avoid row-by-row code such as `iterrows()` unless you truly need custom per-row logic.

# 15. NumPy or pandas?

Use **NumPy** when:

- data is mostly numerical;
- arrays have regular shape;
- vectorized numerical performance matters.

Use **pandas** when:

- data is tabular;
- columns have names;
- data types vary by column;
- grouping, merging, filtering, or missing values are important.

# 16. SciPy: use established scientific algorithms

SciPy provides optimized implementations for:

- optimization;
- interpolation;
- statistics;
- integration;
- signal processing;
- sparse matrices;
- differential equations.

In [63]:
from scipy import optimize

Example: minimize

\[
f(x) = (x - 3)^2 + 2
\]

In [64]:
def f(x):
    return (x - 3)**2 + 2

result = optimize.minimize_scalar(f)

print(result)
print("Minimum near x =", result.x)

 message: 
          Optimization terminated successfully;
          The returned value satisfies the termination criteria
          (using xtol = 1.48e-08 )
 success: True
     fun: 2.0
       x: 3.0
     nit: 4
    nfev: 9
Minimum near x = 3.0


The broader lesson is:

> If a well-tested scientific library already implements the algorithm, use it rather than reproducing it with custom Python loops.

# 17. Sparse matrices

Dense arrays waste memory when most entries are zero.

SciPy provides sparse matrix representations.

In [65]:
from scipy import sparse

dense = np.array([
    [1, 0, 0, 0],
    [0, 0, 5, 0],
    [0, 0, 0, 0],
    [0, 2, 0, 0]
])

S = sparse.csr_matrix(dense)

print(S)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 3 stored elements and shape (4, 4)>
  Coords	Values
  (0, 0)	1
  (1, 2)	5
  (3, 1)	2


Sparse matrices are common in:

- text processing;
- graph problems;
- recommendation systems;
- large linear systems.

# 18. Threads, the GIL, and numerical libraries

For **CPU-bound pure Python code**, multiple threads usually do not give true CPU parallelism because of the CPython GIL.

But this does **not** mean numerical Python is restricted to one core.

Libraries such as:

- NumPy
- SciPy
- PyTorch
- TensorFlow

may execute compiled native code that:

- releases the GIL;
- uses optimized CPU instructions;
- may use multiple native threads internally.

# 19. Multiprocessing

If work cannot be expressed efficiently using NumPy or another optimized library, multiple processes can be useful.

Each process has its own Python interpreter and its own GIL.

Typical `.py`-script pattern:

```python
from multiprocessing import Pool

with Pool(4) as pool:
    results = pool.map(expensive_function, inputs)
```

Multiprocessing works best when:

- the task is CPU-bound;
- tasks are independent;
- each task is relatively expensive;
- communication between processes is small.

## Why chunks help parallelism

Suppose a dataset is split into independent blocks:

```text
chunk 1
chunk 2
chunk 3
chunk 4
```

Sequential execution:

```text
CPU: chunk1 → chunk2 → chunk3 → chunk4
```

Parallel execution:

```text
Core 1: chunk1 → chunk3
Core 2: chunk2 → chunk4
```

Chunking therefore helps both **memory management** and **parallel scheduling**.

# 20. Common anti-patterns

### Anti-pattern 1 — Looping over NumPy arrays

```python
for x in array:
    ...
```

Ask first whether the operation can be written as a vectorized expression.

### Anti-pattern 2 — Looping over DataFrame rows

```python
for _, row in df.iterrows():
    ...
```

Look for a vectorized pandas operation first.

### Anti-pattern 3 — Repeated conversion

```text
list → array → list → array
```

Each conversion costs time and allocates memory.

### Anti-pattern 4 — Huge temporary arrays

Fast code can still be memory inefficient.

Think about:

- `dtype`;
- views;
- in-place operations;
- chunking.

# 21. Mini benchmark summary

In [66]:
N = 2_000_000

start = time.perf_counter()
py_result = sum((i % 7) * (i % 13) for i in range(N))
py_time = time.perf_counter() - start

start = time.perf_counter()
x = np.arange(N)
np_result = np.sum((x % 7) * (x % 13))
np_time = time.perf_counter() - start

print(f"Pure Python: {py_time:.4f} s")
print(f"NumPy      : {np_time:.4f} s")
print(f"Speedup    : {py_time / np_time:.2f}x")
print("Same result:", py_result == np_result)

Pure Python: 0.0896 s
NumPy      : 0.0180 s
Speedup    : 4.97x
Same result: True


# 22. Practical optimization workflow

```text
Write clear correct code
        ↓
Measure
        ↓
Find the bottleneck
        ↓
Can NumPy/pandas/SciPy do it?
        ↓
      yes ──→ use optimized library
        │
        no
        ↓
Can the workload be split into independent large tasks?
        ↓
      yes ──→ multiprocessing
```

The smartest optimization is usually **choosing the right abstraction**, not writing more complicated Python.

# 23. Take-home messages

- Measure before optimizing.
- Prefer NumPy vectorization for numerical array operations.
- Use broadcasting instead of manual repetition.
- Use boolean masks instead of manual filtering.
- Understand `axis` and reductions.
- Choose an appropriate `dtype`.
- Know when slicing returns a view.
- Avoid unnecessary copies and temporary arrays.
- Use chunking for memory-constrained workloads.
- Use pandas for tabular data.
- Use SciPy for established scientific algorithms.
- Use sparse matrices when most entries are zero.
- Native numerical libraries may use multiple cores even though pure Python threads are constrained by the GIL.
- Use multiprocessing when work is CPU-bound, independent, and not easily delegated to an optimized library.